# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [32]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [33]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'

openai = OpenAI()

API key looks good so far


In [34]:
load_dotenv(override=True)
claude_api_key = os.getenv('CLAUDE_API_KEY')
if claude_api_key and claude_api_key.startswith('sk-ant-api03-') and len(claude_api_key)>10:
    print("Claude API key looks good so far")
else:
    print("There might be a problem with your Claude API key? Please visit the troubleshooting notebook!")
    
MODEL = 'claude-sonnet-4-6'
base_url = 'https://api.anthropic.com/v1'

openai = OpenAI(base_url=base_url, api_key=claude_api_key)


Claude API key looks good so far


In [35]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [36]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [37]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [22]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [38]:
def _parse_json_from_response(text):
    """Strip markdown code fences (e.g. ```json ... ```) if present, then parse JSON."""
    if not text or not text.strip():
        raise ValueError("Empty response from model")
    text = text.strip()
    if text.startswith("```"):
        lines = text.split("\n")
        lines = lines[1:]  # drop opening ```json or ```
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        text = "\n".join(lines)
    return json.loads(text)

def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ]
    )

    print(response)
    result = response.choices[0].message.content
    links = _parse_json_from_response(result)
    return links
    

In [39]:
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

In [40]:
select_relevant_links("https://edwarddonner.com")

ChatCompletion(id='msg_01BrW4Na8B1DuyMSFY4UqfoJ', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='```json\n{\n    "links": [\n        {"type": "about page", "url": "https://edwarddonner.com/about-me-and-about-nebula/"},\n        {"type": "curriculum page", "url": "https://edwarddonner.com/curriculum/"},\n        {"type": "posts page", "url": "https://edwarddonner.com/posts/"},\n        {"type": "company page", "url": "https://nebula.io/?utm_source=ed&utm_medium=referral"},\n        {"type": "news article", "url": "https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html"},\n        {"type": "LinkedIn profile", "url": "https://www.linkedin.com/in/eddonner/"},\n        {"type": "project page", "url": "https://edwarddonner.com/outsmart/"},\n        {"type": "project page", "url": "https://edwarddonner.com/connect-four/"},\n        {"type": "proficient page", "url": "https

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'posts page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'news article',
   'url': 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'proficient page', 'url': 'https://edwarddonner.com/proficient/'}]}

In [41]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ]
    )
    result = response.choices[0].message.content
    links = _parse_json_from_response(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [42]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling claude-sonnet-4-6
Found 6 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'blog/posts page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'news article',
   'url': 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html'},
  {'type': 'LinkedIn profile',
   'url': 'https://www.linkedin.com/in/eddonner/'}]}

In [43]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling claude-sonnet-4-6
Found 15 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'models page', 'url': 'https://huggingface.co/models'},
  {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'},
  {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'},
  {'type': 'documentation page', 'url': 'https://huggingface.co/docs'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'community/forum page', 'url': 'https://discuss.huggingface.co'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [53]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    #print(result)
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    print("--------------------------------")
    #print(result)
    return result

In [45]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling claude-sonnet-4-6
Found 19 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
NEW
Storage Buckets: AI-native object storage
GGML and llama.cpp join Hugging Face 🔥
Try HuggingChat Omni – Chat with AI 💬
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Lightricks/LTX-2.3
Updated
7 days ago
•
345k
•
520
Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled
Updated
4 days ago
•
30.8k
•
427
Qwen/Qwen3.5-9B
Updated
10 days ago
•
1.39M
•
734
HauhauCS/Qwen3.5-9B-Uncensored-HauhauCS-Aggressive
Updated
8 days ago
•
127k
•
337
fishaudio/s2-pro
Updated
about 11 hours ago
•
746
•
267
Browse 2M+ models
Spaces
Running
on
Zero
Featured
500
Omni Video Factory
🏆
50

In [46]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [47]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [54]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling claude-sonnet-4-6
Found 15 relevant links
--------------------------------


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nNEW\nStorage Buckets: AI-native object storage\nGGML and llama.cpp join Hugging Face 🔥\nTry HuggingChat Omni – Chat with AI 💬\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nLightricks/LTX-2.3\nUpdated\n7 days ago\n•\n345k\n•\n520\nJackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled\nUpdated\n4 days ago\n•\n30.8k\n•\n428\nQwen/Qwen3.5-9B\nUpdated\n10 days ago\n•\n1.39M\n•\n734\nHauhauCS/Qwen3.5-9B-Uncensored-HauhauCS-Aggressive\

In [55]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )

    print(response)
    result = response.choices[0].message.content
    display(Markdown(result))

In [56]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling claude-sonnet-4-6
Found 15 relevant links
--------------------------------
ChatCompletion(id='msg_018Y673WLDi2VUiRjr7fgQBU', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='# Hugging Face — Company Brochure\n\n---\n\n## 🤗 Who We Are\n\n**Hugging Face** is the home of machine learning — a platform and community where researchers, developers, and organizations collaborate to build, share, and deploy AI models, datasets, and applications. Founded with a mission to **democratize good machine learning, one commit at a time**, Hugging Face has grown into the world\'s leading open-source AI hub.\n\n> *"The AI community building the future."*\n\n---\n\n## 🚀 What We Do\n\nHugging Face provides a comprehensive platform for the entire machine learning lifecycle:\n\n- **Models** — Browse and share from over **2 million+ models**, including cutting-edge large language models, image ge

# Hugging Face — Company Brochure

---

## 🤗 Who We Are

**Hugging Face** is the home of machine learning — a platform and community where researchers, developers, and organizations collaborate to build, share, and deploy AI models, datasets, and applications. Founded with a mission to **democratize good machine learning, one commit at a time**, Hugging Face has grown into the world's leading open-source AI hub.

> *"The AI community building the future."*

---

## 🚀 What We Do

Hugging Face provides a comprehensive platform for the entire machine learning lifecycle:

- **Models** — Browse and share from over **2 million+ models**, including cutting-edge large language models, image generators, audio models, and more.
- **Datasets** — Access and contribute to **500,000+ datasets** for training and evaluation.
- **Spaces** — Discover and deploy **1 million+ AI applications** built by the community.
- **Storage Buckets** — AI-native object storage for your ML assets.
- **HuggingChat** — Chat with state-of-the-art AI models directly through HuggingChat Omni.
- **Open Source Stack** — Powering faster development with tools like Transformers, Diffusers, and now **GGML & llama.cpp**, fully integrated into the platform.

---

## 🌍 Our Platform

Whether you're an individual researcher or a large enterprise, Hugging Face enables you to:

- **Host and collaborate** on unlimited public models, datasets, and applications
- **Move faster** with the HF open-source stack
- **Deploy at scale** with Enterprise-grade solutions
- **Support MCP (Model Context Protocol)** integrations for advanced AI workflows

### Trending This Week
Some of the most exciting models on the platform include:
- **Qwen/Qwen3.5-9B** — 1.39M downloads
- **Lightricks/LTX-2.3** — 345k downloads
- **fishaudio/s2-pro** — Cutting-edge audio generation

---

## 🏢 Enterprise Solutions

Hugging Face offers **Enterprise plans** tailored for organizations that need:
- Private model and dataset hosting
- Advanced security and compliance
- Dedicated support and SLAs
- Scalable inference infrastructure

Trusted by companies across industries to power their AI initiatives with open-source transparency and enterprise reliability.

---

## 📚 Research & Thought Leadership

Hugging Face is an active contributor to AI research and the open-source community. Recent publications and articles include:

- *FineVision: Open Data Is All You Need*
- *SmolVLM: Redefining Small and Efficient Multimodal Models*
- Insights on the global open-source AI ecosystem, including analyses of DeepSeek and shifting compute landscapes

The company regularly publishes **blog posts, guides, case studies**, and **community articles** covering NLP, Computer Vision, Audio, Reinforcement Learning, Diffusion Models, and AI Ethics.

---

## 🌱 Company Culture

Hugging Face is built around **openness, collaboration, and community**. With a team of **183+ members** and tens of thousands of community contributors worldwide, the culture reflects:

- A deep commitment to **open-source principles**
- A belief that **good AI should be accessible to everyone**
- An inclusive, globally distributed team
- Active engagement with the community through blog articles, research papers, and collaborative projects

---

## 💼 Careers & Join Us

Hugging Face is growing and always looking for passionate people who believe in democratizing AI.

> *"If that sounds like something you should be doing, why don't you join us!"*

Whether you're an ML engineer, researcher, developer advocate, or product designer, Hugging Face offers the opportunity to work at the frontier of AI — openly and collaboratively.

📩 **For press inquiries:** Contact the team directly via the Hugging Face website.
🔗 **Explore opportunities:** [huggingface.co](https://huggingface.co)

---

## 📊 At a Glance

| Metric | Detail |
|---|---|
| Models Available | 2,000,000+ |
| Datasets Available | 500,000+ |
| AI Applications | 1,000,000+ |
| Team Members | 183+ |
| Community Followers | 85,000+ |

---

*Hugging Face — Building the future of AI, together.* 🤗

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [58]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [59]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling claude-sonnet-4-6
Found 17 relevant links
--------------------------------


# Hugging Face — Company Brochure

---

## 🤗 Who We Are

**Hugging Face** is the world's leading AI collaboration platform — the home of machine learning. Founded on the mission to **democratize good machine learning, one commit at a time**, Hugging Face provides the tools, infrastructure, and community for researchers, developers, and organizations to build, share, and discover AI models and applications.

> *"The AI community building the future."*

---

## 🚀 What We Offer

### The Platform
Hugging Face hosts the largest open AI ecosystem in the world:

- **2M+ Models** — Browse, upload, and collaborate on machine learning models across every domain
- **500K+ Datasets** — Discover and share datasets for training and evaluation
- **1M+ AI Applications (Spaces)** — Deploy and explore interactive AI-powered apps
- **Storage Buckets** — AI-native object storage for your data and outputs

### Key Products & Features
- **HuggingChat Omni** — Chat with cutting-edge AI models directly
- **Open Source Stack** — Including the widely-adopted Transformers library, DistilBERT, and integrations with tools like llama.cpp & GGML
- **Enterprise Plan** — Advanced security, SSO, audit logs, region control, and dedicated support
- **Team Plan** — Starting at just $20/user/month

---

## 👥 Our Customers

Hugging Face serves a broad and diverse audience:

- **Individual researchers and developers** building and fine-tuning models
- **Startups and scale-ups** leveraging open-source AI to move faster
- **Enterprise organizations** requiring advanced access controls, compliance, and dedicated support
- **Academic institutions and the open-source community** collaborating on the future of AI

Trending community models include contributions from **Qwen, Lightricks, Fish Audio**, and many others — reflecting a truly global user base.

---

## 🌍 Company Culture

Hugging Face is deeply rooted in **openness, collaboration, and community**. Core cultural values include:

- 🔓 **Open Source First** — Committed to making AI accessible to everyone
- 🤝 **Community-Driven** — With over **85,700 followers** and hundreds of active team members and contributors
- 📚 **Education & Thought Leadership** — Publishing articles, papers, and learning resources (including partnerships with platforms like DataCamp)
- 🌐 **Global Perspective** — Actively engaging with and shaping the global AI ecosystem, from DeepSeek to open-source frontiers

---

## 📰 Recent Highlights

- **GGML and llama.cpp** join the Hugging Face ecosystem
- Launch of **Storage Buckets** — AI-native object storage
- **HuggingChat Omni** released for multimodal AI conversations
- Research papers: *FineVision*, *SmolVLM*, and more
- New articles exploring the global open-source AI landscape

---

## 💼 Careers & Joining the Team

Hugging Face is a growing team of **183+ members** passionate about AI and its democratization. If you believe in building open, accessible, and powerful machine learning tools for everyone, Hugging Face wants to hear from you.

> *"If that sounds like something you should be doing, why don't you join us!"*

🔗 Visit [huggingface.co](https://huggingface.co) to explore open roles and become part of the AI community building the future.

---

## 📬 Get in Touch

- **General:** [huggingface.co](https://huggingface.co)
- **Enterprise Sales:** Contact the sales team for flexible contract options
- **Press Enquiries:** Contact via the official press team on the website

---

*Hugging Face — Democratizing AI, one commit at a time.* 🤗

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>